In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from falsb4mpa.dataset.utils import bucket

In [2]:
used_columns = [
    'sex', #Bin
    'age', #Bin
    'race', #Bin
    'juv_fel_count', #MinMax
    'juv_misd_count', #MinMax
    'juv_other_count', #MinMax
    'priors_count', #MinMax
    'c_charge_degree', #Bin
    'is_recid', #Already bin
    'is_violent_recid', #Already bin
    'two_year_recid' #Already bin
]
target = 'decile_score' # set high_risk

# Reading data

In [3]:
raw_data = pd.read_csv("../../../data/raw/compas/compas.csv", index_col=0)

In [4]:
raw_data['high_risk'] = (raw_data['decile_score'] >= 7).astype(int)

In [5]:
cols = used_columns + ['high_risk']
data = raw_data[cols]

In [6]:
data.head()

,sex,age,race,juv_fel_count,juv_misd_count,juv_other_count,priors_count,c_charge_degree,is_recid,is_violent_recid,two_year_recid,high_risk
id,,,,,,,,,,,,
1,Male,69,Other,0,0,0,0,F,0,0,0,0
3,Male,34,African-American,0,0,0,0,F,1,1,1,0
4,Male,24,African-American,0,0,1,4,F,1,0,1,0
5,Male,23,African-American,0,1,0,1,F,0,0,0,1
6,Male,43,Other,0,0,0,2,F,0,0,0,0


In [7]:
print(len(data.index))

7214


In [8]:
data.isna().sum()

sex                 0
age                 0
race                0
juv_fel_count       0
juv_misd_count      0
juv_other_count     0
priors_count        0
c_charge_degree     0
is_recid            0
is_violent_recid    0
two_year_recid      0
high_risk           0
dtype: int64

# Binarizing

In [9]:
data['sex'].value_counts()

sex
Male      5819
Female    1395
Name: count, dtype: int64

In [10]:
data['sex'] = (data['sex'] == 'Male').astype(int)
data['sex'].value_counts()

/var/folders/nx/ktkztyln3xn8f8j2qrfpjrlh0000gn/T/ipykernel_14467/1893676450.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['sex'] = (data['sex'] == 'Male').astype(int)


sex
1    5819
0    1395
Name: count, dtype: int64

In [11]:
data[data['age'] < 25]

,sex,age,race,juv_fel_count,juv_misd_count,juv_other_count,priors_count,c_charge_degree,is_recid,is_violent_recid,two_year_recid,high_risk
id,,,,,,,,,,,,
4,1,24,African-American,0,0,1,4,F,1,0,1,0
5,1,23,African-American,0,1,0,1,F,0,0,0,1
13,1,21,Caucasian,0,0,0,1,F,1,1,1,0
15,1,23,African-American,0,0,0,3,M,1,0,1,0
26,1,21,African-American,0,0,0,1,F,1,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...
10992,1,21,Caucasian,0,0,0,0,M,1,0,1,0
10995,1,20,African-American,0,0,0,0,F,0,0,0,1
10996,1,23,African-American,0,0,0,0,F,0,0,0,1


In [12]:
data['age'] = (data['age'] < 25).astype(int)
data['age'].value_counts()

/var/folders/nx/ktkztyln3xn8f8j2qrfpjrlh0000gn/T/ipykernel_14467/208118988.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['age'] = (data['age'] < 25).astype(int)


age
0    5685
1    1529
Name: count, dtype: int64

In [13]:
data['race'].value_counts()

race
African-American    3696
Caucasian           2454
Hispanic             637
Other                377
Asian                 32
Native American       18
Name: count, dtype: int64

In [14]:
data['race'] = (data['race'] == 'Caucasian').astype(int)
data['race'].value_counts()

/var/folders/nx/ktkztyln3xn8f8j2qrfpjrlh0000gn/T/ipykernel_14467/2615921338.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['race'] = (data['race'] == 'Caucasian').astype(int)


race
0    4760
1    2454
Name: count, dtype: int64

In [15]:
data['c_charge_degree'].value_counts()

c_charge_degree
F    4666
M    2548
Name: count, dtype: int64

In [16]:
data['c_charge_degree'] = (data['c_charge_degree'] == 'M').astype(int)
data['c_charge_degree'].value_counts()

/var/folders/nx/ktkztyln3xn8f8j2qrfpjrlh0000gn/T/ipykernel_14467/1280120774.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['c_charge_degree'] = (data['c_charge_degree'] == 'M').astype(int)


c_charge_degree
0    4666
1    2548
Name: count, dtype: int64

# MinMax scaler

In [17]:
continous_attr = [
    'juv_fel_count', #MinMax
    'juv_misd_count', #MinMax
    'juv_other_count', #MinMax
    'priors_count', #MinMax
]

scaler = MinMaxScaler()

In [18]:
for attr in continous_attr:
    data[attr] = scaler.fit_transform(np.array(data[attr]).reshape(-1,1))

/var/folders/nx/ktkztyln3xn8f8j2qrfpjrlh0000gn/T/ipykernel_14467/3382165602.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[attr] = scaler.fit_transform(np.array(data[attr]).reshape(-1,1))
/var/folders/nx/ktkztyln3xn8f8j2qrfpjrlh0000gn/T/ipykernel_14467/3382165602.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[attr] = scaler.fit_transform(np.array(data[attr]).reshape(-1,1))
/var/folders/nx/ktkztyln3xn8f8j2qrfpjrlh0000gn/T/ipykernel_14467/3382165602.py:2: SettingWithCopyWarning: 
A value i

In [19]:
data.describe()

,sex,age,race,juv_fel_count,juv_misd_count,juv_other_count,priors_count,c_charge_degree,is_recid,is_violent_recid,two_year_recid,high_risk
count,7214.000000,7214.000000,7214.000000,7214.000000,7214.000000,7214.000000,7214.000000,7214.000000,7214.000000,7214.000000,7214.000000,7214.000000
mean,0.806626,0.211949,0.340172,0.003362,0.006995,0.006434,0.091379,0.353202,0.481148,0.113529,0.450652,0.276546
std,0.394971,0.408717,0.473800,0.023699,0.037326,0.029505,0.128488,0.477998,0.499679,0.317261,0.497593,0.447321
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.052632,0.000000,0.000000,0.000000,0.000000,0.000000
75%,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.131579,1.000000,1.000000,0.000000,1.000000,1.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


# Reordering the columns

In [20]:
columns_order = [
    'sex', #Bin
    'race', #Bin
    'age', #Bin
    'juv_fel_count', #MinMax
    'juv_misd_count', #MinMax
    'juv_other_count', #MinMax
    'priors_count', #MinMax
    'c_charge_degree', #Bin
    'is_recid', #Already bin
    'is_violent_recid', #Already bin
    'two_year_recid', #Already bin
    'high_risk'
]

In [21]:
data.head()

,sex,age,race,juv_fel_count,juv_misd_count,juv_other_count,priors_count,c_charge_degree,is_recid,is_violent_recid,two_year_recid,high_risk
id,,,,,,,,,,,,
1,1,0,0,0.0,0.000000,0.000000,0.000000,0,0,0,0,0
3,1,0,0,0.0,0.000000,0.000000,0.000000,0,1,1,1,0
4,1,1,0,0.0,0.000000,0.058824,0.105263,0,1,0,1,0
5,1,1,0,0.0,0.076923,0.000000,0.026316,0,0,0,0,1
6,1,0,0,0.0,0.000000,0.000000,0.052632,0,0,0,0,0


In [22]:
data = data[columns_order]
data.head()

,sex,race,age,juv_fel_count,juv_misd_count,juv_other_count,priors_count,c_charge_degree,is_recid,is_violent_recid,two_year_recid,high_risk
id,,,,,,,,,,,,
1,1,0,0,0.0,0.000000,0.000000,0.000000,0,0,0,0,0
3,1,0,0,0.0,0.000000,0.000000,0.000000,0,1,1,1,0
4,1,0,1,0.0,0.000000,0.058824,0.105263,0,1,0,1,0
5,1,0,1,0.0,0.076923,0.000000,0.026316,0,0,0,0,1
6,1,0,0,0.0,0.000000,0.000000,0.052632,0,0,0,0,0


# Saving data

In [23]:
print(len(data.index))

7214


In [24]:
data.to_csv('../../../data/processed/compas/compas_mpa_bin_wout_agg.csv')